<img src="rupixen-Q59HmzK38eQ-unsplash.jpg" alt="Someone is trying to purchase a produce online" width="500"/>

Online shopping decisions rely on how consumers engage with online store content. You work for a new startup company that has just launched a new online shopping website. The marketing team asks you, a new data scientist, to review a dataset of online shoppers' purchasing intentions gathered over the last year. Specifically, the team wants you to generate some insights into customer browsing behaviors in November and December, the busiest months for shoppers. You have decided to identify two groups of customers: those with a low purchase rate and returning customers. After identifying these groups, you want to determine the probability that any of these customers will make a purchase in a new marketing campaign to help gauge potential success for next year's sales.

### Data description:

You are given an `online_shopping_session_data.csv` that contains several columns about each shopping session. Each shopping session corresponded to a single user. 

|Column|Description|
|--------|-----------|
|`SessionID`|unique session ID|
|`Administrative`|number of pages visited related to the customer account|
|`Administrative_Duration`|total amount of time spent (in seconds) on administrative pages|
|`Informational`|number of pages visited related to the website and the company|
|`Informational_Duration`|total amount of time spent (in seconds) on informational pages|
|`ProductRelated`|number of pages visited related to available products|
|`ProductRelated_Duration`|total amount of time spent (in seconds) on product-related pages|
|`BounceRates`|average bounce rate of pages visited by the customer|
|`ExitRates`|average exit rate of pages visited by the customer|
|`PageValues`|average page value of pages visited by the customer|
|`SpecialDay`|closeness of the site visiting time to a specific special day|
|`Weekend`|indicator whether the session is on a weekend|
|`Month`|month of the session date|
|`CustomerType`|customer type|
|`Purchase`|class label whether the customer make a purchase|

In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load and view your data
shopping_data = pd.read_csv("online_shopping_session_data.csv")
shopping_data.head()

,SessionID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Weekend,Month,CustomerType,Purchase
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,False,Feb,Returning_Customer,0.0
1,2,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,False,Feb,Returning_Customer,0.0
2,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,False,Feb,Returning_Customer,0.0
3,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,False,Feb,Returning_Customer,0.0
4,5,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,True,Feb,Returning_Customer,0.0


In [9]:
# Start your code here!
# Use as many cells as you like
shopping_data_nov_dec = shopping_data[(shopping_data["Month"] == "Nov") | (shopping_data["Month"] == "Dec")]
shopping_data_nov_dec.head()
customer_purchase_counts = shopping_data_nov_dec.groupby(["CustomerType","Purchase"]).size()
total_new = np.sum(customer_purchase_counts["New_Customer"])
total_returning = np.sum(customer_purchase_counts["Returning_Customer"])
print(customer_purchase_counts)
purchases_new = customer_purchase_counts.get(("New_Customer", 1), 0)
purchases_returning = customer_purchase_counts.get(("Returning_Customer", 1), 0)
purchase_rate_new = purchases_new / total_new
purchase_rate_returning = purchases_returning / total_returning
purchase_rates = {
    "Returning_Customer": purchase_rate_returning,
    "New_Customer": purchase_rate_new
}
print(purchase_rates)

CustomerType        Purchase
New_Customer        0.0          529
                    1.0          199
Returning_Customer  0.0         2994
                    1.0          728
dtype: int64
{'Returning_Customer': 0.1955937667920473, 'New_Customer': 0.2733516483516483}


In [10]:
returning_nov_dec = shopping_data[
    (shopping_data["CustomerType"] == "Returning_Customer") &
    (shopping_data["Month"].isin(["Nov", "Dec"]))
]
duration_cols = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]
durations = returning_nov_dec[duration_cols]
# Example approach
cor_admin_info = durations["Administrative_Duration"].corr(durations["Informational_Duration"])
cor_admin_prod = durations["Administrative_Duration"].corr(durations["ProductRelated_Duration"])
cor_info_prod = durations["Informational_Duration"].corr(durations["ProductRelated_Duration"])
print(cor_admin_info, cor_admin_prod , cor_info_prod)
top_correlation = {
    "pair": ("Administrative_Duration", "ProductRelated_Duration"),
    "correlation": 0.41689293883422757
}

0.25810799265722995 0.41689293883422757 0.36655111971168536


In [11]:
p_new = 1.15 * purchase_rate_returning
from scipy.stats import binom

prob_at_least_100_sales = 1 - binom.cdf(99, 500, p_new)